# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset with automated reporting, EDA, and model interpretation.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas jinja2 shap
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# @title 3. Automated EDA (YData Profiling)
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="AutoML Pilot EDA Report", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize, interpret_model as cls_interpret
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize, interpret_model as reg_interpret

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]
use_gpu = False # @param {type:"boolean"}

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False, use_gpu=use_gpu)
    print("🚀 Comparing models...")
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False, use_gpu=use_gpu)
    print("🚀 Comparing models...")
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

print("✅ Training Complete!")
display(leaderboard.head())

In [ ]:
# @title 5. Model Interpretation (SHAP)
try:
    if task_type == 'classification':
        cls_interpret(final_model)
    else:
        reg_interpret(final_model)
except Exception as e:
    print(f"⚠️ Interpretation not supported for this model type: {e}")

In [ ]:
# @title 6. Professional Email Reporting
import smtplib, ssl, json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"🚀 AutoML Pilot Pro - {task_type.capitalize()} Report"
    msg["From"] = f"AutoML Pilot <{sender_email}>"
    msg["To"] = recipient_email

    metrics_html = leaderboard.head(5).to_html(classes='table table-striped', border=0)
    
    html_body = f"""
    <html>
    <body style='font-family: Arial, sans-serif; background-color: #f4f4f9; padding: 20px;'>
        <div style='max-width: 600px; margin: auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); border-top: 5px solid #7c3aed;'>
            <h1 style='color: #7c3aed; text-align: center;'>AutoML Pilot Pro</h1>
            <p style='color: #4b5563; font-size: 16px;'>Your training session on Google Colab has finished successfully.</p>
            
            <div style='background: #f9fafb; padding: 15px; border-radius: 8px; margin: 20px 0;'>
                <h3 style='margin-top: 0; color: #1f2937;'>📊 Top Models</h3>
                {metrics_html}
            </div>
            
            <p style='color: #6b7280; font-size: 14px;'>Download your <b>best_model.pkl</b> and upload it to the <a href="https://automlpilot.streamlit.app">AutoML Pilot Pro</a> deployment tab for instant inference.</p>
            
            <hr style='border: 0; border-top: 1px solid #e5e7eb; margin: 20px 0;'>
            <p style='text-align: center; color: #9ca3af; font-size: 12px;'>Sent via AutoML Pilot Pro Colab Integration</p>
        </div>
    </body>
    </html>
    """
    
    msg.attach(MIMEText(html_body, "html"))
    context = ssl.create_default_context()
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print('✅ Professional email report sent!')
    except Exception as e:
        print(f'❌ Email failed: {e}')

In [ ]:
# @title 7. Download Model & Metadata
from google.colab import files
import json

if os.path.exists('best_model.pkl'):
    # Save metadata
    metadata = {
        "target": target_column,
        "task": task_type,
        "features": list(df.columns.drop(target_column)),
        "source": "Google Colab"
    }
    with open('model_metadata.json', 'w') as f:
        json.dump(metadata, f)
    
    files.download('best_model.pkl')
    files.download('model_metadata.json')
    print('✅ Model and metadata downloaded! Ready for deployment.')